# vLLM and Qdrant: hands-on

Plain HTTP calls to the two services from `docker-compose.yml`, so you can see the raw APIs.
Start them first: `docker compose up -d`.

- **vLLM** serves an OpenAI-compatible API on `:8000`
- **Qdrant** serves a REST API on `:6333` (dashboard: http://localhost:6333/dashboard)

In [1]:
import json

import httpx

from rag.config import load_config

cfg = load_config()
VLLM = cfg.llm.base_url   # http://localhost:8000/v1
MODEL = cfg.llm.model     # "llm" = --served-model-name
QDRANT = cfg.qdrant.url   # http://localhost:6333

## vLLM

### Which model is loaded
`GET /v1/models` — `id` is the name clients use, `root` is the Hugging Face repo behind it.

### Chat completion
`POST /v1/chat/completions` — same request shape as OpenAI. The model applies its chat template to `messages`,
generates up to `max_tokens`, and reports token counts in `usage`. `temperature=0` makes the output (nearly) deterministic.

In [4]:
def chat(prompt: str, **params) -> dict:
    body = {"model": MODEL, "messages": [{"role": "user", "content": prompt}], "max_tokens": 64, "temperature": 0, **params}
    return httpx.post(f"{VLLM}/chat/completions", json=body, timeout=60).json()

resp = chat("What is the capital of France? Answer in one word.")
print(resp["choices"][0]["message"]["content"])
print(resp["usage"])

Paris
{'prompt_tokens': 41, 'total_tokens': 43, 'completion_tokens': 2, 'prompt_tokens_details': None}


In [5]:
resp = chat("Какая столица Франции? Ответь одним словом.")
print(resp["choices"][0]["message"]["content"])
print(resp["usage"])

Париж
{'prompt_tokens': 45, 'total_tokens': 49, 'completion_tokens': 4, 'prompt_tokens_details': None}


### System prompt
A `system` message sets the rules for the conversation. This is how the RAG prompt (Phase 6) will tell the model
to answer only from the provided context.

In [6]:
body = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "Answer only from the context. If the answer is not there, say 'not found'.\n\nContext: [1] Nikola Tesla died on 7 January 1943."},
        {"role": "user", "content": "When did Tesla die? And where was he born?"},
    ],
    "max_tokens": 64,
    "temperature": 0,
}
print(httpx.post(f"{VLLM}/chat/completions", json=body, timeout=60).json()["choices"][0]["message"]["content"])

Tesla died on 7 January 1943. The context does not provide information about where Tesla was born.


### Streaming
With `"stream": true` the server sends SSE (Server-Sent Events): one `data: {...}` line per chunk of tokens,
each holding a `delta`, then `data: [DONE]`. This is what makes time-to-first-token short.

In [7]:
body = {"model": MODEL, "messages": [{"role": "user", "content": "Расскажи в двух предложениях, что такое RAG."}],
        "max_tokens": 128, "temperature": 0, "stream": True}

with httpx.stream("POST", f"{VLLM}/chat/completions", json=body, timeout=60) as r:
    for line in r.iter_lines():
        if not line.startswith("data: "):
            continue
        data = line.removeprefix("data: ")
        if data == "[DONE]":
            break
        print(json.loads(data)["choices"][0]["delta"].get("content", ""), end="", flush=True)

RAG — это методика или архитектура для моделирования и использования релевантных контекстных данных, где "R" означает Retrieval (извлечение), "A" — Augmentation (расширение), а "G" — Generation (генерация). Этот подход позволяет моделям машинного обучения использовать внешние данные для улучшения точности и релевантности ответов.

## Qdrant

### Server status
`GET /collections` lists collections. Fresh install → empty list.

In [8]:
print(httpx.get(QDRANT).json())               # version info
httpx.get(f"{QDRANT}/collections").json()

{'title': 'qdrant - vector search engine', 'version': '1.13.4', 'commit': '7abc684361fb81d8b62cf1554d8bf4fb65a706d7'}


{'result': {'collections': []}, 'status': 'ok', 'time': 3.1e-06}

### Create a collection
A collection is like a table. Each point has an `id`, one or more **named vectors**, and a JSON **payload**.
Here: one named vector `dense` of size 4 with cosine distance (the real one will be 1024-dim bge-m3 vectors).
Creating a collection that already exists is an error, so we drop it first.

In [9]:
COL = f"{QDRANT}/collections/demo"
httpx.delete(COL)
httpx.put(COL, json={"vectors": {"dense": {"size": 4, "distance": "Cosine"}}}).json()

{'result': True, 'status': 'ok', 'time': 0.137684899}

### Insert points
`wait=true` returns only after the points are indexed and searchable.
The `acl` payload field mimics the permission groups the real index will store per chunk.

In [10]:
points = [
    {"id": 1, "vector": {"dense": [0.9, 0.1, 0.0, 0.0]}, "payload": {"text": "Tesla died in 1943", "acl": ["all-employees"]}},
    {"id": 2, "vector": {"dense": [0.8, 0.2, 0.1, 0.0]}, "payload": {"text": "Tesla was born in Smiljan", "acl": ["hr-team"]}},
    {"id": 3, "vector": {"dense": [0.0, 0.1, 0.9, 0.3]}, "payload": {"text": "The Rhine flows into the North Sea", "acl": ["all-employees"]}},
]
httpx.put(f"{COL}/points", params={"wait": "true"}, json={"points": points}).json()

{'result': {'operation_id': 0, 'status': 'completed'},
 'status': 'ok',
 'time': 0.001046181}

### Vector search
`POST /points/query` returns the nearest points by cosine similarity (`score`, higher = closer).

In [11]:
query = [1.0, 0.1, 0.0, 0.0]   # "about Tesla"

def search(**extra):
    body = {"query": query, "using": "dense", "limit": 3, "with_payload": True, **extra}
    for p in httpx.post(f"{COL}/points/query", json=body).json()["result"]["points"]:
        print(f"{p['score']:.3f}  {p['payload']}")

search()

1.000  {'text': 'Tesla died in 1943', 'acl': ['all-employees']}
0.982  {'text': 'Tesla was born in Smiljan', 'acl': ['hr-team']}
0.010  {'text': 'The Rhine flows into the North Sea', 'acl': ['all-employees']}


### Search with an ACL filter
The filter is applied **inside** the search, not after it, so `limit` results are all ones the user may see.
`match.any` = the point's `acl` shares at least one group with the user's groups.
A point without an `acl` field never matches — that's the fail-closed behavior the plan relies on.

In [12]:
alice_groups = ["all-employees"]
search(filter={"must": [{"key": "acl", "match": {"any": alice_groups}}]})

1.000  {'text': 'Tesla died in 1943', 'acl': ['all-employees']}
0.010  {'text': 'The Rhine flows into the North Sea', 'acl': ['all-employees']}


### Update payload without touching the vector
This is the cheap path for an ACL-only change: no re-embedding.

In [13]:
httpx.post(f"{COL}/points/payload", params={"wait": "true"}, json={"payload": {"acl": ["all-employees"]}, "points": [2]}).json()
search(filter={"must": [{"key": "acl", "match": {"any": alice_groups}}]})

1.000  {'text': 'Tesla died in 1943', 'acl': ['all-employees']}
0.982  {'text': 'Tesla was born in Smiljan', 'acl': ['all-employees']}
0.010  {'text': 'The Rhine flows into the North Sea', 'acl': ['all-employees']}


### Clean up

In [14]:
httpx.delete(COL).json()

{'result': True, 'status': 'ok', 'time': 0.003964591}